# 01 · Load S&P 500 five-minute prices and historical membership

This notebook loads the local published Alpaca data and the matching historical
membership list, audits date/security coverage, checks missing regular-session
bars, and displays an OHLCV sample. It reads the existing files without changing
prices, resampling, filling gaps, or fetching cloud data.

**Default scope:** the full available history, using **raw** prices. Change the
parameters below to inspect a smaller interval or the adjusted series. Prices
remain in DuckDB views; only summaries and one stock/day are loaded into pandas.

The source notebook has no saved outputs. Execution results, plots, samples and
an executed notebook are kept under the Git-ignored `data/reports/` directory.

**Workspace review:** source notebooks contain no private outputs. Model training and feature rebuilding are disabled by default. Read `docs/AUDIT_FINDINGS.md` before interpreting results.

Install dependencies once from the project root with `.venv/bin/python -m pip install -r requirements.txt`.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
from zoneinfo import ZoneInfo
import json
import os

ADJUSTMENT = "raw"            # "raw" or "all"; audit one series at a time
START_DATE = None              # e.g. "2020-01-01"; None uses the release boundary
END_DATE = None                # e.g. "2020-12-31"
SAMPLE_TICKER = "AAPL"
SAMPLE_DATE = None             # None selects its latest observed member-day

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_DIR = next((p for p in candidates
    if (p / "README.md").is_file()
    and (p / "data/sp500_5min_20261103/MANIFEST.json").is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Run this notebook from the 5min_momentum project folder.")

PRICE_DIR = PROJECT_DIR / "data/sp500_5min_20261103"
PIT_DIR = PROJECT_DIR / "data/sp500_pit_20261103"
REPORT_DIR = PROJECT_DIR / "data/reports/01_load_sp500_5min"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(REPORT_DIR / ".matplotlib"))

import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import exchange_calendars as xcals
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 18)
pd.set_option("display.max_rows", 25)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.2})
print(f"Project: {PROJECT_DIR}")
print(f"Private reports: {REPORT_DIR.relative_to(PROJECT_DIR)}")

## 1. Release, file coverage and important limits

The S&P 500 copy is selected from the larger multi-index release; their sizes
overlap. Both `raw` and `all` adjustments are present, so combining them would
double-count bars. This audit selects one adjustment.

Membership is retrospective with date-level availability. After March 2026 its
S&P 500 extension uses Sharadar change dates that may precede the effective
session by up to a week. Price history covers **index-member days only**.

In [ ]:
price_manifest = json.loads((PRICE_DIR / "MANIFEST.json").read_text())
pit_manifest = json.loads((PIT_DIR / "MANIFEST.json").read_text())
assert price_manifest["complete"], "The local price download/preparation is incomplete."
assert ADJUSTMENT in price_manifest["price_adjustments"]

release_start = pd.Timestamp(price_manifest["membership"]["start"])
release_end = pd.Timestamp(price_manifest["membership"]["finish"])
scope_start = pd.Timestamp(START_DATE) if START_DATE else release_start
scope_end = pd.Timestamp(END_DATE) if END_DATE else release_end
assert release_start <= scope_start <= scope_end <= release_end, "Choose dates inside the release."
start_iso, end_iso = scope_start.date().isoformat(), scope_end.date().isoformat()

inventory_rows = []
for adjustment in ("raw", "all"):
    paths = sorted((PRICE_DIR / f"adjustment={adjustment}").glob("month=*/*.parquet"))
    for path in paths:
        metadata = pq.ParquetFile(path).metadata
        inventory_rows.append({"adjustment": adjustment, "month": path.parent.name.split("=", 1)[1],
            "rows": metadata.num_rows, "size_bytes": path.stat().st_size, "path": str(path)})
inventory = pd.DataFrame(inventory_rows)
assert len(inventory) == price_manifest["expected_partitions"]
assert inventory["rows"].sum() == price_manifest["bars"]
assert inventory["size_bytes"].sum() == price_manifest["bytes"]
assert not inventory.duplicated(["adjustment", "month"]).any()

file_coverage = inventory.groupby("adjustment").agg(
    files=("path", "size"), first_month=("month", "min"), last_month=("month", "max"),
    bars=("rows", "sum"), size_bytes=("size_bytes", "sum")).reset_index()
file_coverage["size_GB"] = file_coverage.pop("size_bytes") / 1e9
display(file_coverage)
print(f"Audit: {ADJUSTMENT} prices, {start_iso} through {end_iso}")
print("Bar counts above describe the full release; scoped counts are calculated below.")
display(pd.DataFrame({"source_caveat": price_manifest["known_issues"]}))

## 2. Load prices and both membership tables

`bars` is a lazy view of the selected price partitions. `pit_security` has unique
security/date keys suitable for joins. `pit_series` retains all source membership
series, including unresolved identities and conflicting series. Never derive the
historical universe solely from the presence of price bars.

In [ ]:
month_start, month_end = start_iso[:7], end_iso[:7]
selected_files = inventory.loc[
    inventory["adjustment"].eq(ADJUSTMENT)
    & inventory["month"].between(month_start, month_end)
].sort_values("month")
assert len(selected_files), "No local partitions in this scope."

con = duckdb.connect()
con.execute("SET threads=4")
con.execute("SET memory_limit='2GB'")
con.execute("SET TimeZone='UTC'")
spill_dir = REPORT_DIR / ".duckdb_tmp"
spill_dir.mkdir(exist_ok=True)
con.execute("SET temp_directory = ?", [str(spill_dir)])

con.from_parquet(selected_files["path"].tolist(), hive_partitioning=False).create_view("bars_source")
con.from_parquet(str(PIT_DIR / "membership_by_security_date.parquet")).create_view("pit_security_source")
con.from_parquet(str(PIT_DIR / "membership_daily.parquet")).create_view("pit_series_source")
# ISO dates are validated above; SQL literals here contain only those dates.
con.execute(f"""CREATE VIEW bars AS SELECT * FROM bars_source
    WHERE trading_date BETWEEN '{start_iso}' AND '{end_iso}'""")
for name in ("pit_security", "pit_series"):
    con.execute(f"""CREATE VIEW {name} AS SELECT * FROM {name}_source
        WHERE trading_date BETWEEN DATE '{start_iso}' AND DATE '{end_iso}'""")

display(con.sql("DESCRIBE SELECT * FROM bars").df())
display(con.sql("SELECT * FROM pit_series ORDER BY trading_date, membership_series_id LIMIT 5").df())
display(con.sql("""SELECT 'membership series/date' AS grain, count(*) AS member_rows,
    count(DISTINCT security_id) AS resolved_security_ids,
    min(trading_date) AS first_date, max(trading_date) AS last_date,
    count_if(security_id IS NULL OR security_id='') AS unresolved_rows
    FROM pit_series
    UNION ALL
    SELECT 'unique security/date', count(*), count(DISTINCT security_id),
    min(trading_date), max(trading_date), 0 FROM pit_security""").df())

pit_duplicate_keys = con.sql("""SELECT count(*) AS duplicate_keys FROM (
    SELECT security_id,trading_date FROM pit_security GROUP BY ALL HAVING count(*)>1)""").fetchall()[0][0]
assert pit_duplicate_keys == 0, "A PIT join would multiply price rows."
gap_reasons = con.sql("""SELECT coalesce(price_gap_reason,'priced member row') AS status,
    count(*) AS membership_series_days FROM pit_series GROUP BY 1 ORDER BY 2 DESC""").df()
display(gap_reasons)

## 3. Build the expected regular-session calendar

Use the XNYS exchange calendar as the US cash-equity session reference. A normal
session has **78** five-minute slots; a 13:00 early close has **42**. Bars are
start-labelled and the close is exclusive: 09:30, 09:35, …, 15:55 on a normal day.

Extended-hours bars are counted, but no complete extended-hours grid is assumed.
The source's clock-based `is_regular_session` flag is checked against this calendar
rather than used to define it. Membership dates outside this calendar are reported.

In [ ]:
calendar = xcals.get_calendar("XNYS", start=start_iso, end=end_iso)
calendar_df = calendar.schedule.loc[start_iso:end_iso, ["open", "close"]].reset_index()
calendar_df.columns = ["trading_date", "market_open_utc", "market_close_utc"]
duration_seconds = (calendar_df["market_close_utc"] - calendar_df["market_open_utc"]).dt.total_seconds()
assert (duration_seconds % 300 == 0).all()
calendar_df["expected_5min_bars"] = (duration_seconds / 300).astype("int64")
calendar_df["early_close"] = calendar_df["expected_5min_bars"] < 78
con.register("calendar_frame", calendar_df)
con.execute("""CREATE VIEW sessions AS SELECT CAST(trading_date AS DATE) AS trading_date,
    market_open_utc, market_close_utc, expected_5min_bars, early_close FROM calendar_frame""")
print(f"{len(calendar_df):,} sessions; {calendar_df['early_close'].sum():,} early closes.")
display(calendar_df.loc[calendar_df["early_close"]].tail(10))
non_session_members = con.sql("""SELECT p.* FROM pit_series p
    ANTI JOIN sessions s USING(trading_date)""").df()
print(f"Membership-series rows outside the reference calendar: {len(non_session_members):,}")
if len(non_session_members):
    display(non_session_members.head(10))

## 4. Scan the price history once, month by month

Monthly aggregation keeps memory bounded. Missing slots use **distinct valid
five-minute timestamps**, so duplicate bars cannot inflate coverage. We also count
timestamp/availability errors, OHLC inconsistencies and source regular-session
flag differences.

A present five-minute bar can contain only 1–4 minute bars. That is a separate
diagnostic: absent minutes may reflect no eligible trades, a halt or a data gap.
It is not automatic evidence of a bad download.

In [ ]:
scan_started = perf_counter()
con.execute("DROP TABLE IF EXISTS observed")
for number, row in enumerate(selected_files.itertuples(index=False), 1):
    con.from_parquet(row.path, hive_partitioning=False).create_view("month_source", replace=True)
    audit_sql = f"""
    WITH typed AS (
        SELECT security_id,CAST(trading_date AS DATE) AS trading_date,ticker,
            try_cast(bar_timestamp_utc AS TIMESTAMPTZ) AS ts,
            try_cast(available_at_utc AS TIMESTAMPTZ) AS available_at,
            is_regular_session,minute_bars,open,high,low,close,volume,trade_count
        FROM month_source WHERE trading_date BETWEEN '{start_iso}' AND '{end_iso}'
    ), flags AS (
        SELECT b.*,s.market_open_utc,s.market_close_utc,
            coalesce(ts>=s.market_open_utc AND ts<s.market_close_utc,false) AS in_rth,
            coalesce(ts>=s.market_open_utc AND ts<s.market_close_utc
                AND epoch(ts)::BIGINT % 300=0,false) AS in_expected_slot
        FROM typed b LEFT JOIN sessions s USING(trading_date)
    )
    SELECT security_id,trading_date,min(ticker) AS price_ticker,
        count(*) AS all_bars,count(ts)-count(DISTINCT ts) AS duplicate_timestamp_rows,
        count_if(in_rth) AS rth_rows,
        count(DISTINCT CASE WHEN in_expected_slot THEN ts END) AS observed_rth_slots,
        count_if(in_expected_slot AND minute_bars<5) AS short_minute_bucket_rows,
        count_if(in_rth AND volume=0) AS zero_volume_rth_rows,
        count_if(is_regular_session IS DISTINCT FROM in_rth) AS session_flag_disagreements,
        count_if(ts IS NULL OR available_at IS NULL) AS invalid_timestamps,
        count_if(ts IS NOT NULL AND epoch(ts)::BIGINT % 300!=0) AS misaligned_timestamps,
        count_if(available_at IS NOT NULL AND ts IS NOT NULL
            AND available_at!=ts+INTERVAL '5 minutes') AS wrong_availability,
        count_if(minute_bars IS NULL OR minute_bars NOT BETWEEN 1 AND 5) AS invalid_minute_counts,
        count_if(open IS NULL OR high IS NULL OR low IS NULL OR close IS NULL) AS null_ohlc_rows,
        count_if(high<low OR open<low OR open>high OR close<low OR close>high) AS invalid_ohlc_rows
    FROM flags GROUP BY security_id,trading_date
    """
    statement = "CREATE TEMP TABLE observed AS " if number == 1 else "INSERT INTO observed "
    con.execute(statement + audit_sql)
    if number == 1 or number % 12 == 0 or number == len(selected_files):
        print(f"Scanned {number:3d}/{len(selected_files)} months; {perf_counter()-scan_started:,.1f}s elapsed")

observed_totals = con.sql("""SELECT count(*) AS security_days, count(DISTINCT security_id) AS securities,
    min(trading_date) AS first_bar_date,max(trading_date) AS last_bar_date,
    sum(all_bars) AS all_session_bars,sum(observed_rth_slots) AS regular_session_slots,
    sum(short_minute_bucket_rows) AS rth_bars_with_fewer_than_5_minutes,
    sum(duplicate_timestamp_rows) AS duplicate_timestamp_rows,
    sum(session_flag_disagreements) AS clock_flag_disagreements,
    sum(invalid_timestamps) AS invalid_timestamps,sum(misaligned_timestamps) AS misaligned_timestamps,
    sum(wrong_availability) AS wrong_availability,sum(invalid_minute_counts) AS invalid_minute_counts,
    sum(null_ohlc_rows) AS null_ohlc_rows,sum(invalid_ohlc_rows) AS invalid_ohlc_rows FROM observed""").df()
display(observed_totals.T.rename(columns={0:"observed"}))
if START_DATE is None and END_DATE is None:
    assert int(observed_totals.iloc[0]["all_session_bars"]) == price_manifest["bars_per_adjustment"][ADJUSTMENT]

outside_pit = con.sql("""SELECT o.* FROM observed o
    ANTI JOIN pit_security p USING(security_id,trading_date)""").df()
print(f"Observed security/dates outside the resolved PIT list: {len(outside_pit):,}")

## 5. Coverage and missing regular-session bars

The main denominator is **all resolved PIT security/dates**, including days with
no daily price. A second coverage measure considers only days with a daily price.
Unresolved identities are reported separately, because their stock IDs cannot be
joined safely. Conflicting membership series are deduplicated at security/date.

`missing_rth_slots = expected calendar slots − distinct observed valid slots`.
This identifies absent bars; it does not determine their cause or justify filling them.

In [ ]:
con.execute("""CREATE TEMP TABLE coverage AS
    SELECT p.security_id,p.trading_date,p.membership_tickers,p.has_daily_price,
        p.has_conflicting_membership_series,p.price_gap_reasons,
        coalesce(o.price_ticker,array_to_string(p.membership_tickers,',')) AS ticker,
        s.expected_5min_bars,s.early_close,
        coalesce(o.all_bars,0) AS all_bars,coalesce(o.observed_rth_slots,0) AS observed_rth_slots,
        CASE WHEN s.expected_5min_bars IS NOT NULL
          THEN s.expected_5min_bars-coalesce(o.observed_rth_slots,0) END AS missing_rth_slots,
        coalesce(o.short_minute_bucket_rows,0) AS short_minute_bucket_rows,
        coalesce(o.session_flag_disagreements,0) AS session_flag_disagreements,
        CASE WHEN s.trading_date IS NULL THEN 'outside_reference_calendar'
          WHEN NOT p.has_daily_price THEN 'no_daily_price'
          WHEN coalesce(o.all_bars,0)=0 THEN 'no_5min_bars'
          WHEN coalesce(o.observed_rth_slots,0)=0 THEN 'extended_hours_only'
          WHEN o.observed_rth_slots=s.expected_5min_bars THEN 'complete_regular_session'
          ELSE 'partial_regular_session' END AS coverage_status
    FROM pit_security p LEFT JOIN observed o USING(security_id,trading_date)
    LEFT JOIN sessions s USING(trading_date)""")
assert con.sql("SELECT count(*) FROM coverage").fetchall()[0][0] == con.sql("SELECT count(*) FROM pit_security").fetchall()[0][0]
assert con.sql("SELECT count(*) FROM coverage WHERE missing_rth_slots<0").fetchall()[0][0] == 0
published_count_mismatches = con.sql(f"""SELECT count(*) FROM coverage c
    JOIN pit_security p USING(security_id,trading_date)
    WHERE c.all_bars!=p.{ADJUSTMENT}_bars_5min""").fetchall()[0][0]
assert published_count_mismatches == 0, "Bar counts differ from the published member-day table."

coverage_status = con.sql("""SELECT coverage_status,count(*) AS security_days,
    sum(expected_5min_bars) AS expected_slots,sum(observed_rth_slots) AS observed_slots,
    sum(missing_rth_slots) AS missing_slots FROM coverage GROUP BY 1 ORDER BY 2 DESC""").df()
display(coverage_status)

con.execute("""CREATE TEMP TABLE coverage_by_day AS
    SELECT trading_date,count(*) AS resolved_member_days,
      count_if(has_daily_price) AS days_with_daily_price,
      count_if(observed_rth_slots>0) AS days_with_regular_bars,
      count_if(missing_rth_slots>0) AS member_days_with_missing_slots,
      sum(expected_5min_bars) AS expected_slots,sum(observed_rth_slots) AS observed_slots,
      sum(missing_rth_slots) AS missing_slots,
      sum(CASE WHEN NOT has_daily_price THEN missing_rth_slots ELSE 0 END) AS missing_on_no_daily_price_days,
      sum(CASE WHEN has_daily_price THEN missing_rth_slots ELSE 0 END) AS missing_on_priced_days,
      sum(CASE WHEN has_daily_price THEN expected_5min_bars ELSE 0 END) AS expected_on_priced_days,
      sum(short_minute_bucket_rows) AS short_minute_bucket_rows
    FROM coverage WHERE expected_5min_bars IS NOT NULL GROUP BY 1 ORDER BY 1""")
coverage_by_day = con.sql("""SELECT *,100.0*observed_slots/nullif(expected_slots,0) AS coverage_pct,
    100.0*observed_slots/nullif(expected_on_priced_days,0) AS priced_day_coverage_pct
    FROM coverage_by_day ORDER BY trading_date""").df()

for period, expression in [("month", "date_trunc('month',trading_date)"), ("year", "year(trading_date)")]:
    con.execute(f"""CREATE TEMP TABLE coverage_by_{period} AS
      SELECT {expression} AS {period},count(*) AS sessions,
        sum(expected_slots) AS expected_slots,sum(observed_slots) AS observed_slots,
        sum(missing_slots) AS missing_slots,
        sum(missing_on_no_daily_price_days) AS missing_on_no_daily_price_days,
        sum(missing_on_priced_days) AS missing_on_priced_days,
        sum(expected_on_priced_days) AS expected_on_priced_days,
        100.0*sum(observed_slots)/nullif(sum(expected_slots),0) AS coverage_pct,
        100.0*sum(observed_slots)/nullif(sum(expected_on_priced_days),0) AS priced_day_coverage_pct
      FROM coverage_by_day GROUP BY 1 ORDER BY 1""")
coverage_by_month = con.sql("SELECT * FROM coverage_by_month ORDER BY month").df()
coverage_by_year = con.sql("SELECT * FROM coverage_by_year ORDER BY year").df()
display(coverage_by_year)

unresolved_members = con.sql("""SELECT * FROM pit_series
    WHERE security_id IS NULL OR security_id=''""").df()
print(f"Unresolved membership-series days, outside the resolved-security denominator: {len(unresolved_members):,}")
if len(unresolved_members):
    display(con.sql("""SELECT membership_ticker,company_name,count(*) AS unresolved_member_days,
        min(trading_date) AS first_date,max(trading_date) AS last_date
        FROM pit_series WHERE security_id IS NULL OR security_id=''
        GROUP BY 1,2 ORDER BY 3 DESC LIMIT 10""").df())

display(con.sql("""SELECT security_id,ticker,trading_date,coverage_status,
    expected_5min_bars,observed_rth_slots,missing_rth_slots,price_gap_reasons
    FROM coverage WHERE missing_rth_slots>0
    ORDER BY missing_rth_slots DESC,trading_date DESC LIMIT 15""").df())

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,4.2),layout="constrained")
axes[0].plot(coverage_by_month["month"],coverage_by_month["coverage_pct"],
    color="#1d4f78",label="All resolved member-days")
axes[0].plot(coverage_by_month["month"],coverage_by_month["priced_day_coverage_pct"],
    color="#d17b29",label="Member-days with daily prices")
axes[0].set(title="Regular-session slot coverage",ylabel="Observed / expected (%)")
axes[0].xaxis.set_major_locator(mdates.YearLocator(2))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[0].legend(frameon=False,fontsize=8)
x = np.arange(len(coverage_by_year))
axes[1].bar(x,coverage_by_year["missing_on_no_daily_price_days"]/1e6,
    color="#b7c8d8",label="No daily price")
axes[1].bar(x,coverage_by_year["missing_on_priced_days"]/1e6,
    bottom=coverage_by_year["missing_on_no_daily_price_days"]/1e6,
    color="#d17b29",label="Daily price exists")
axes[1].set_xticks(x,coverage_by_year["year"].astype(str),rotation=45)
axes[1].set(title="Absent expected five-minute bars",ylabel="Missing slots (millions)")
axes[1].legend(frameon=False,fontsize=8)
fig.savefig(REPORT_DIR / "coverage_overview.png",bbox_inches="tight")
plt.show()

## 6. Inspect exact missing timestamps

Select a member-day with observed regular-session bars and missing slots, where
available; otherwise show a fully absent day. The expected grid is joined to the
actual timestamps, so the table shows the exact missing intervals, not an assumed
row-count difference. These bars are left missing.

In [ ]:
missing_case = con.sql("""SELECT security_id,ticker,trading_date,has_daily_price,
    observed_rth_slots,missing_rth_slots FROM coverage WHERE missing_rth_slots>0
    ORDER BY (observed_rth_slots>0) DESC,has_daily_price DESC,
        missing_rth_slots DESC,trading_date DESC LIMIT 1""").df()
missing_example = pd.DataFrame()
if len(missing_case):
    chosen = missing_case.iloc[0]
    case_date = pd.Timestamp(chosen["trading_date"]).date().isoformat()
    case_path = PRICE_DIR / f"adjustment={ADJUSTMENT}/month={case_date[:7]}/part-00000.parquet"
    con.from_parquet(str(case_path),hive_partitioning=False).create_view("example_source",replace=True)
    missing_example = con.execute("""
      WITH expected AS (
        SELECT unnest(generate_series(market_open_utc,market_close_utc-INTERVAL '5 minutes',
            INTERVAL '5 minutes')) AS bar_timestamp_utc
        FROM sessions WHERE trading_date=CAST(? AS DATE)
      ), actual AS (
        SELECT try_cast(bar_timestamp_utc AS TIMESTAMPTZ) AS bar_timestamp_utc
        FROM example_source WHERE security_id=? AND trading_date=?
      )
      SELECT e.bar_timestamp_utc,
        timezone('America/New_York',e.bar_timestamp_utc) AS bar_timestamp_new_york
      FROM expected e ANTI JOIN actual a USING(bar_timestamp_utc) ORDER BY 1
      """,[case_date,str(chosen["security_id"]),case_date]).df()
    assert len(missing_example)==int(chosen["missing_rth_slots"])
    display(missing_case)
    print(f"{len(missing_example)} exact missing five-minute slots (first 20 shown):")
    display(missing_example.head(20))
else:
    print("No absent regular-session slots in this scope.")

## 7. Show a five-minute OHLCV sample

The sample uses the selected adjustment and an actual historical member-day.
Only that stock/day is loaded into pandas. Its regular session uses the exchange
calendar, including an early close if applicable. Price gaps are not filled.

In [ ]:
sample_candidates = con.execute("""SELECT security_id,trading_date,price_ticker AS ticker
    FROM observed WHERE price_ticker=? AND observed_rth_slots>0
      AND (? IS NULL OR trading_date=try_cast(? AS DATE))
    ORDER BY trading_date DESC LIMIT 1""",[SAMPLE_TICKER,SAMPLE_DATE,SAMPLE_DATE]).df()
if sample_candidates.empty:
    raise ValueError("No regular-session member-day for this sample ticker/date in the selected scope.")
selected_sample = sample_candidates.iloc[0]
sample_date = pd.Timestamp(selected_sample["trading_date"]).date().isoformat()
sample_id = str(selected_sample["security_id"])
sample_path = PRICE_DIR / f"adjustment={ADJUSTMENT}/month={sample_date[:7]}/part-00000.parquet"
con.from_parquet(str(sample_path),hive_partitioning=False).create_view("sample_source",replace=True)
sample = con.execute("""SELECT b.security_id,b.ticker,b.adjustment,
    CAST(b.bar_timestamp_utc AS TIMESTAMPTZ) AS bar_timestamp_utc,
    timezone('America/New_York',CAST(b.bar_timestamp_utc AS TIMESTAMPTZ)) AS bar_timestamp_new_york,
    CAST(b.available_at_utc AS TIMESTAMPTZ) AS available_at_utc,
    b.open,b.high,b.low,b.close,b.volume,b.trade_count,b.vwap,b.minute_bars
    FROM sample_source b JOIN sessions s ON CAST(b.trading_date AS DATE)=s.trading_date
    WHERE b.security_id=? AND b.trading_date=?
      AND CAST(b.bar_timestamp_utc AS TIMESTAMPTZ)>=s.market_open_utc
      AND CAST(b.bar_timestamp_utc AS TIMESTAMPTZ)<s.market_close_utc
    ORDER BY bar_timestamp_utc""",[sample_id,sample_date]).df()
assert not sample.empty and sample["security_id"].eq(sample_id).all()
sample["minutes_since_previous_bar"] = sample["bar_timestamp_utc"].diff().dt.total_seconds()/60
sample_slots = con.execute("""SELECT expected_5min_bars,observed_rth_slots,missing_rth_slots
    FROM coverage WHERE security_id=? AND trading_date=CAST(? AS DATE)""",[sample_id,sample_date]).df()
print(f"{SAMPLE_TICKER}, {sample_date}, adjustment={ADJUSTMENT}, security_id={sample_id}")
display(sample_slots)
display(sample.head(12))

fig,axes = plt.subplots(2,1,figsize=(12,5.4),sharex=True,
    height_ratios=[3,1],layout="constrained")
axes[0].plot(sample["bar_timestamp_utc"],sample["close"],color="#1d4f78",marker=".",markersize=3)
axes[0].set(title=f"{SAMPLE_TICKER} · {sample_date} · {ADJUSTMENT} five-minute bars",ylabel="Close (USD)")
axes[1].bar(sample["bar_timestamp_utc"],sample["volume"],width=4/(24*60),color="#8aa9be")
axes[1].set(ylabel="Volume",xlabel="New York time")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M",tz=ZoneInfo("America/New_York")))
fig.savefig(REPORT_DIR / "sample_5min.png",bbox_inches="tight")
plt.show()

## 8. Save private audit outputs

Coverage summaries, membership exceptions, the sample and charts stay under
`data/reports/01_load_sp500_5min/`. Rerunning with a different scope replaces this
notebook's report files; the selected parameters are recorded in the JSON summary.
The underlying published prices and PIT files are never modified.

Before using the data for a strategy: investigate missing bars and identity gaps,
account for retrospective membership timing, and handle adjustment-vintage changes.

In [ ]:
for table in ("coverage","coverage_by_day","coverage_by_month","coverage_by_year"):
    con.table(table).write_parquet(str(REPORT_DIR / f"{table}.parquet"),compression="zstd")
observed_totals.to_parquet(REPORT_DIR / "observed_totals.parquet",index=False)
unresolved_members.to_parquet(REPORT_DIR / "unresolved_members.parquet",index=False)
non_session_members.to_parquet(REPORT_DIR / "non_session_members.parquet",index=False)
sample.to_parquet(REPORT_DIR / "sample_5min.parquet",index=False)
if not missing_example.empty:
    missing_example.to_parquet(REPORT_DIR / "missing_example_slots.parquet",index=False)

totals = con.sql("""SELECT sum(expected_5min_bars),sum(observed_rth_slots),sum(missing_rth_slots),
    count_if(missing_rth_slots>0),count_if(NOT has_daily_price)
    FROM coverage WHERE expected_5min_bars IS NOT NULL""").fetchall()[0]
summary = {
    "adjustment":ADJUSTMENT,"scope_start":start_iso,"scope_end":end_iso,
    "calendar":"XNYS","sessions":len(calendar_df),
    "all_session_bars":int(observed_totals.iloc[0]["all_session_bars"]),
    "observed_securities":int(observed_totals.iloc[0]["securities"]),
    "expected_rth_slots_resolved_members":int(totals[0]),
    "observed_rth_slots":int(totals[1]),"missing_rth_slots":int(totals[2]),
    "resolved_member_days_with_missing_slots":int(totals[3]),
    "resolved_member_days_without_daily_prices":int(totals[4]),
    "regular_slot_coverage_pct":100*int(totals[1])/int(totals[0]),
    "priced_day_regular_slot_coverage_pct":100*int(coverage_by_year["observed_slots"].sum())/int(coverage_by_year["expected_on_priced_days"].sum()),
    "unresolved_membership_series_days":len(unresolved_members),
    "non_session_membership_series_days":len(non_session_members),
    "bars_outside_pit_security_dates":len(outside_pit),
    "published_member_day_count_mismatches":int(published_count_mismatches),
    "rth_bars_with_fewer_than_5_source_minutes":int(observed_totals.iloc[0]["rth_bars_with_fewer_than_5_minutes"]),
    "quality_checks":{name:int(observed_totals.iloc[0][name]) for name in (
        "duplicate_timestamp_rows","clock_flag_disagreements","invalid_timestamps",
        "misaligned_timestamps","wrong_availability","invalid_minute_counts",
        "null_ohlc_rows","invalid_ohlc_rows")},
    "sample":{"ticker":SAMPLE_TICKER,"date":sample_date,"security_id":sample_id,"bars":len(sample)},
    "created_at_utc":datetime.now(timezone.utc).isoformat(),
}
(REPORT_DIR / "SUMMARY.json").write_text(json.dumps(summary,indent=2)+"\n")
display(pd.DataFrame({"metric":summary.keys(),"value":[str(v) for v in summary.values()]}))
print(f"Private results saved to: {REPORT_DIR.relative_to(PROJECT_DIR)}")